In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Data load karein
df = pd.read_csv(r"C:\Users\Mithlesh Maurya\Desktop\Project\final_mega_sales_data_10k.csv")
df['Order_Date'] = pd.to_datetime(df['Order_Date'])


from statsmodels.tsa.holtwinters import ExponentialSmoothing

# Dictionary banate hain results store karne ke liye
all_forecasts = {}
summary_list = []

# Har category ke liye loop chalayenge
for cat in df['Category'].unique():
    # 1. Data filter aur daily resample
    cat_series = df[df['Category'] == cat].set_index('Order_Date')['Total_Sales'].resample('D').sum().fillna(0)
    
    # 2. Model Train (Exponential Smoothing)
    # Agar data kam hai toh simple model use karenge
    try:
        model = ExponentialSmoothing(cat_series, trend='add', seasonal='add', seasonal_periods=7).fit()
        forecast = model.forecast(7) # Agle 7 din ki prediction
        
        # 3. Insights nikalna
        current_avg = cat_series.tail(7).mean()
        forecast_avg = forecast.mean()
        status = "Rising 📈" if forecast_avg > current_avg else "Falling 📉"
        
        summary_list.append({
            'Category': cat,
            'Current_Avg_Sales': round(current_avg, 2),
            'Predicted_Avg_Sales': round(forecast_avg, 2),
            'Trend': status
        })
        all_forecasts[cat] = forecast
    except:
        continue

# Summary DataFrame dikhayye
summary_df = pd.DataFrame(summary_list).sort_values(by='Predicted_Avg_Sales', ascending=False)
print("--- Sales Trend Summary for All Categories ---")
print(summary_df.to_string(index=False))



c:\ProgramData\anaconda3\Lib\site-packages\statsmodels\tsa\holtwinters\model.py:918: ConvergenceWarning: Optimization failed to converge. Check mle_retvals.
  warnings.warn(
c:\ProgramData\anaconda3\Lib\site-packages\statsmodels\tsa\holtwinters\model.py:918: ConvergenceWarning: Optimization failed to converge. Check mle_retvals.
  warnings.warn(
c:\ProgramData\anaconda3\Lib\site-packages\statsmodels\tsa\holtwinters\model.py:918: ConvergenceWarning: Optimization failed to converge. Check mle_retvals.
  warnings.warn(
c:\ProgramData\anaconda3\Lib\site-packages\statsmodels\tsa\holtwinters\model.py:918: ConvergenceWarning: Optimization failed to converge. Check mle_retvals.
  warnings.warn(
c:\ProgramData\anaconda3\Lib\site-packages\statsmodels\tsa\holtwinters\model.py:918: ConvergenceWarning: Optimization failed to converge. Check mle_retvals.
  warnings.warn(
c:\ProgramData\anaconda3\Lib\site-packages\statsmodels\tsa\holtwinters\model.py:918: ConvergenceWarning: Optimization failed to co

--- Sales Trend Summary for All Categories ---
              Category  Current_Avg_Sales  Predicted_Avg_Sales     Trend
           Electronics          591448.43            521118.64 Falling 📉
          Fitness Gear          138902.57            114609.24 Falling 📉
             Furniture           41295.71             36739.14 Falling 📉
        Women Clothing           20308.57             33875.40  Rising 📈
          Toys & Games           22023.14             32592.78  Rising 📈
            Automotive           42345.86             32478.07 Falling 📉
         Kitchen Tools           31200.00             25101.50 Falling 📉
              Footwear           22897.00             19730.62 Falling 📉
Beauty & Personal Care           23219.43             15640.63 Falling 📉
            Stationery            5093.14             12332.96  Rising 📈
            Home Decor           13644.29             12114.76 Falling 📉
          Men Clothing           13035.00              9569.25 Falling 📉
    

c:\ProgramData\anaconda3\Lib\site-packages\statsmodels\tsa\holtwinters\model.py:918: ConvergenceWarning: Optimization failed to converge. Check mle_retvals.
  warnings.warn(


In [8]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
import pandas as pd

# 1. Categories aur Cities ko numbers mein badlein
le_cat = LabelEncoder()
le_city = LabelEncoder()
le_channel = LabelEncoder()

df['Cat_Code'] = le_cat.fit_transform(df['Category'])
df['City_Code'] = le_city.fit_transform(df['City'])
df['Channel_Code'] = le_channel.fit_transform(df['Sales_Channel'])

# 2. Demand Level define karein (Training ke liye)
# Units_Sold ko 3 parts mein divide karenge: Low, Medium, High
df['Demand_Level'] = pd.qcut(df['Units_Sold'], 3, labels=['Low', 'Medium', 'High'])

# 3. Features (X) select karein (Rating ki jagah Stock aur Channel use kiya hai)
X = df[['Cat_Code', 'City_Code', 'Unit_Price', 'Stock_Available', 'Channel_Code']]
y = df['Demand_Level']

# 4. Model Train karein
clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X, y)

# 5. Prediction Column add karein
df['Predicted_Demand_Level'] = clf.predict(X)

# Faltu numeric columns hata kar final file save karein
final_df = df.drop(['Cat_Code', 'City_Code', 'Channel_Code'], axis=1)
final_df.to_csv('demand_prediction_no_rating.csv', index=False)

print("Naya column 'Predicted_Demand_Level' bina rating ke add ho gaya hai!")

Naya column 'Predicted_Demand_Level' bina rating ke add ho gaya hai!


In [9]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder

# 1. Encoding (Category aur City ko numbers mein badlein)
le_cat = LabelEncoder()
le_city = LabelEncoder()
df['Cat_Code'] = le_cat.fit_transform(df['Category'])
df['City_Code'] = le_city.fit_transform(df['City'])

# 2. Features (X) aur Target (y)
# Hum Category, Price aur Units_Sold ke basis par Channel predict karenge
X = df[['Cat_Code', 'Unit_Price', 'Units_Sold', 'City_Code']]
y = df['Sales_Channel'] # Target: Online ya Offline

# 3. Model Training
channel_clf = RandomForestClassifier(n_estimators=100, random_state=42)
channel_clf.fit(X, y)

# 4. Naya Column: Predicted Best Channel
df['Suggested_Channel'] = channel_clf.predict(X)

# 5. Ek aur advance column: Probability (Kitne % chance hai Online hone ka)
# Isse aap "Confidence Score" dikha sakte hain dashboard mein
probs = channel_clf.predict_proba(X)
df['Channel_Confidence_Score'] = probs.max(axis=1).round(2)

# Final CSV save karein
df.drop(['Cat_Code', 'City_Code'], axis=1).to_csv('final_pro_dataset.csv', index=False)
print("Naya Column 'Suggested_Channel' add ho gaya hai!")

Naya Column 'Suggested_Channel' add ho gaya hai!
